In [1]:
import torch
import numpy as np
from datasets import Dataset
from transformers import pipeline, AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import accuracy_score, f1_score, classification_report
from tqdm.auto import tqdm


In [ ]:
from tablevault import tablevault
import os

vault = tablevault.Vault(user_id="jinjin",
                            process_name="hf_pipeline_mrpc_margin_rule",
                            arango_url="http://localhost:8629",
                            arango_db="tv_experiment_1",
                            arango_username="tablevault_user",
                            arango_password="tablevault_password",
                            new_arango_db=False,               
                            arango_root_username="root",
                            arango_root_password="passwd",
                            description_embedding_size=3072,
                        )

from openai import OpenAI


openai_key_file = "/Users/jinjinzhao/Documents/work_projects/my_keys/my_keys/openai_jinjin.key"
with open(openai_key_file, 'r') as f:
    openai_key = f.read()

os.environ["OPENAI_API_KEY"] = openai_key

client = OpenAI()

In [ ]:
def get_embeddings(text):
    return client.embeddings.create(
            input=text,
            model="text-embedding-3-large"
        ).data[0].embedding

In [2]:
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print("device:", device)


device: mps


In [3]:
model_name = "textattack/distilbert-base-uncased-MRPC"
margin_threshold = 0.15

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name).to(device)
model.eval()

id2label = model.config.id2label
label0 = id2label[0]
label1 = id2label[1]

clf = pipeline(
    "text-classification",
    model=model,
    tokenizer=tokenizer,
    top_k=None,
    function_to_apply="softmax",
    device=device,
)

print(model_name)
print(id2label)
print("margin_threshold:", margin_threshold)


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

textattack/distilbert-base-uncased-MRPC
{0: 'LABEL_0', 1: 'LABEL_1'}
margin_threshold: 0.15


In [4]:
ds = vault.query_item_content("glue_mrpc_validation")
ds = Dataset.from_dict(ds)
print(ds)
print(ds[0])


Dataset({
    features: ['sentence1', 'sentence2', 'label', 'idx'],
    num_rows: 408
})
{'sentence1': "He said the foodservice pie business doesn 't fit the company 's long-term growth strategy .", 'sentence2': '" The foodservice pie business does not fit our long-term growth strategy .', 'label': 1, 'idx': 9}


In [5]:
sent1 = ds["sentence1"]
sent2 = ds["sentence2"]
y_true = np.array(ds["label"])
examples = [{"text": s1, "text_pair": s2} for s1, s2 in zip(sent1, sent2)]

print("num_examples:", len(y_true))
print("positive_rate:", y_true.mean())
print("example_record:", examples[0])


num_examples: 408
positive_rate: 0.6838235294117647
example_record: {'text': "He said the foodservice pie business doesn 't fit the company 's long-term growth strategy .", 'text_pair': '" The foodservice pie business does not fit our long-term growth strategy .'}


In [6]:
batch_size = 64
score_neg = []
score_pos = []

with torch.no_grad():
    for i in tqdm(range(0, len(examples), batch_size)):
        batch_examples = examples[i:i + batch_size]
        outputs = clf(batch_examples, batch_size=batch_size, truncation=True, max_length=128)
        for out in outputs:
            score_map = {item["label"]: item["score"] for item in out}
            score_neg.append(float(score_map[label0]))
            score_pos.append(float(score_map[label1]))

score_neg = np.array(score_neg)
score_pos = np.array(score_pos)
margins = score_pos - score_neg
y_pred = (margins > margin_threshold).astype(int)

print("done")
print("score arrays:", score_neg.shape, score_pos.shape, margins.shape)


  0%|          | 0/7 [00:00<?, ?it/s]

done
score arrays: (408,) (408,) (408,)


In [ ]:

vault.create_record_list("distilbert_margin_score_prediction_values", column_names=["prediction", "score_pos", "score_neg"])

for i in range(len(y_pred)):
    vault.append_record("distilbert_margin_score_prediction_values", 
                        {
                            "prediction": y_pred[i],
                            "score_pos": float(score_pos[i]),
                            "score_neg": float(score_neg[i]),
                        },
                       input_items = {
                           "glue_mrpc_validation": [i, i + 1],
                       }
                       )

description = "INSERT TEXT HERE ABOUT distilbert_margin_score_prediction_values"
embedding = get_embeddings(description)
vault.create_description("distilbert_margin_score_prediction_values", description, embedding)

properties = {"INSERT_PROPERTY": "INSERT_CATEGORIES"} #e.g. task: paraphrase detection

for prop, cat in properties.items():
    embedding = get_embeddings(prop)
    vault.create_description("distilbert_margin_score_prediction_values", cat, embedding, prop)

In [7]:
acc = accuracy_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)
report = classification_report(y_true, y_pred, target_names=["not_paraphrase", "paraphrase"])
print({"accuracy": acc, "f1": f1})
print(classification_report(y_true, y_pred, target_names=["not_paraphrase", "paraphrase"]))


{'accuracy': 0.8627450980392157, 'f1': 0.9044368600682594}
                precision    recall  f1-score   support

not_paraphrase       0.86      0.67      0.76       129
    paraphrase       0.86      0.95      0.90       279

      accuracy                           0.86       408
     macro avg       0.86      0.81      0.83       408
  weighted avg       0.86      0.86      0.86       408



In [8]:
for i in range(5):
    print("=" * 80)
    print("sentence1:", sent1[i])
    print("sentence2:", sent2[i])
    print(
        "true:", int(y_true[i]),
        "pred:", int(y_pred[i]),
        "neg_score:", round(float(score_neg[i]), 6),
        "pos_score:", round(float(score_pos[i]), 6),
        "margin:", round(float(margins[i]), 6),
        "label:", id2label[int(y_pred[i])],
    )


sentence1: He said the foodservice pie business doesn 't fit the company 's long-term growth strategy .
sentence2: " The foodservice pie business does not fit our long-term growth strategy .
true: 1 pred: 1 neg_score: 0.016485 pos_score: 0.983515 margin: 0.967029 label: LABEL_1
sentence1: Magnarelli said Racicot hated the Iraqi regime and looked forward to using his long years of training in the war .
sentence2: His wife said he was " 100 percent behind George Bush " and looked forward to using his years of training in the war .
true: 0 pred: 0 neg_score: 0.81763 pos_score: 0.18237 margin: -0.635259 label: LABEL_0
sentence1: The dollar was at 116.92 yen against the yen , flat on the session , and at 1.2891 against the Swiss franc , also flat .
sentence2: The dollar was at 116.78 yen JPY = , virtually flat on the session , and at 1.2871 against the Swiss franc CHF = , down 0.1 percent .
true: 0 pred: 0 neg_score: 0.746699 pos_score: 0.253301 margin: -0.493398 label: LABEL_0
sentence1: T

In [9]:
mistakes = np.where(y_true != y_pred)[0][:10]
print("num_errors:", int((y_true != y_pred).sum()))

for i in mistakes:
    print("=" * 80)
    print("idx:", int(i))
    print("sentence1:", sent1[i])
    print("sentence2:", sent2[i])
    print(
        "true:", int(y_true[i]),
        "pred:", int(y_pred[i]),
        "neg_score:", round(float(score_neg[i]), 6),
        "pos_score:", round(float(score_pos[i]), 6),
        "margin:", round(float(margins[i]), 6),
    )


num_errors: 56
idx: 6
sentence1: While dioxin levels in the environment were up last year , they have dropped by 75 percent since the 1970s , said Caswell .
sentence2: The Institute said dioxin levels in the environment have fallen by as much as 76 percent since the 1970s .
true: 0 pred: 1 neg_score: 0.070466 pos_score: 0.929534 margin: 0.859068
idx: 26
sentence1: Cooley said he expects Muhammad will similarly be called as a witness at a pretrial hearing for Malvo .
sentence2: Lee Boyd Malvo will be called as a witness Wednesday in a pretrial hearing for fellow sniper suspect John Allen Muhammad .
true: 0 pred: 1 neg_score: 0.157475 pos_score: 0.842525 margin: 0.685051
idx: 35
sentence1: Bush wanted " to see an aircraft landing the same way that the pilots saw an aircraft landing , " White House press secretary Ari Fleischer said yesterday .
sentence2: On Tuesday , before Byrd 's speech , Fleischer said Bush wanted ' ' to see an aircraft landing the same way that the pilots saw an airc

In [10]:

vault.create_record_list("hf_pipeline_mrpc_margin_rule_summary", column_names=["accuracy", "f1", "classification_report"])


summary = {
    "accuracy": float(acc),
    "f1": float(f1),
    "classification_report": str(report)
}

vault.append_record("hf_pipeline_mrpc_margin_rule_summary", summary,
                    input_items = {
                        "glue_mrpc_validation": [0, len(ds)],
                        "distilbert_margin_score_prediction_values": [0, len(ds)]
                    })

summary

description = "INSERT TEXT HERE ABOUT hf_pipeline_mrpc_margin_rule_summary"
embedding = get_embeddings(description)
vault.create_description("hf_pipeline_mrpc_margin_rule_summary", description, embedding)

properties = {"INSERT_PROPERTY": "INSERT_CATEGORIES"} #e.g. task: paraphrase detection

for prop, cat in properties.items():
    embedding = get_embeddings(prop)
    vault.create_description("hf_pipeline_mrpc_margin_rule_summary", cat, embedding, prop)



{'dataset': 'glue/mrpc',
 'split': 'validation',
 'model': 'textattack/distilbert-base-uncased-MRPC',
 'device': 'mps',
 'margin_threshold': 0.15,
 'num_examples': 408,
 'accuracy': 0.8627450980392157,
 'f1': 0.9044368600682594}

In [ ]:
description = "INSERT TEXT HERE ABOUT hf_pipeline_mrpc_margin_rule process/notebook" # description of whole notebook
embedding = get_embeddings(description)
vault.create_description("hf_pipeline_mrpc_margin_rule", description, embedding)

properties = {"INSERT_PROPERTY": "INSERT_CATEGORIES"} #e.g. model: distilbert-base-uncased-MRPC

for prop, cat in properties.items():
    embedding = get_embeddings(prop)
    vault.create_description("hf_pipeline_mrpc_margin_rule", cat, embedding, prop)